# Lição 16 - Implantando Agentes Escaláveis com Microsoft Foundry

Neste notebook, você vai construir um **agente de suporte ao cliente pronto para produção** para a empresa fictícia **Contoso**. Diferente das lições anteriores, o foco aqui não é o ciclo de raciocínio do agente — é tudo que envolve *ao redor* dele que torna um agente seguro para rodar em escala:

1. **Chamada de ferramentas** — consultas de pedidos e criação de tickets.
2. **RAG** — respostas de política a partir de uma base de conhecimento.
3. **Memória** — lembrar do cliente durante as interações.
4. **Roteamento de modelo** — enviar solicitações simples para um modelo pequeno, e as complexas para um modelo grande.
5. **Cache de respostas** — atender perguntas repetidas sem chamar o modelo.
6. **Aprovação humana** — reembolsos acima de um limite aguardam aprovação.
7. **Portão de avaliação** — um conjunto de testes offline que bloqueia uma versão ruim.
8. **Observabilidade** — rastreamento OpenTelemetry em todas as solicitações.

Cada seção é independente e executável. Leia cada linha — as primitivas de produção são mantidas propositalmente pequenas.


## Configuração

Antes de executar este notebook, certifique-se de que você tem:

1. **Um projeto Microsoft Foundry** com um modelo de chat implantado (por exemplo, `gpt-5-mini`).
2. **Conectado com o Azure CLI** — execute `az login` no seu terminal.
3. **Definido as variáveis de ambiente necessárias:**
   - `AZURE_AI_PROJECT_ENDPOINT` — o endpoint do seu projeto Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — o nome do seu modelo implantado.

A seção RAG usa **Azure AI Search** quando `AZURE_SEARCH_SERVICE_ENDPOINT` e `AZURE_SEARCH_API_KEY` estão definidos, e recorre a uma busca em memória para que o notebook funcione sem um recurso Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Ferramentas

As ferramentas de produção realizam trabalho real em sistemas reais. Aqui simulamos um banco de dados de pedidos e um sistema de tickets com funções Python simples. O decorador `@tool` os expõe ao agente.

Observe que `issue_refund` usa `approval_mode="always_require"` para reembolsos acima de um limite — este é o primitivo de humano no loop que implementamos depois.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Base de Conhecimento de Políticas

Perguntas sobre políticas ("qual é o seu prazo de devolução?") devem ser respondidas a partir de uma fonte autoritativa, não da memória do modelo. Nós encapsulamos uma pequena base de conhecimento como uma ferramenta de busca.

Em produção, isso é o **Azure AI Search**; aqui fornecemos uma busca por palavra-chave em memória para que o notebook funcione em qualquer lugar, e alternamos para o Azure AI Search automaticamente quando as variáveis de ambiente estiverem presentes.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memória

Um agente de suporte que esquece com quem está falando é um agente de suporte ruim. Nós mantemos um pequeno armazenamento de perfil por cliente e inserimos um resumo curto nas instruções do agente. Em produção, isso é um serviço de memória (veja a Lição 13); aqui um dicionário torna o padrão visível.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Roteamento de Modelo e Cache de Resposta

Dois mecanismos de custo conectados a um único manipulador de requisição:

- **Roteamento**: um classificador heurístico simples decide se uma requisição precisa do modelo pequeno ou do grande.
- **Cache**: perguntas repetidas normalizadas são respondidas diretamente a partir de um cache sem chamada ao modelo.

O classificador aqui é intencionalmente simples. Em produção, você o validaria contra o tráfego e poderia substituí-lo pelo Model Router do Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 e 8. O Agente, Aprovação Humana e Observabilidade

Agora montamos o agente a partir das ferramentas acima e envolvemos cada solicitação em uma span do OpenTelemetry. A função `handle_support_request` é o manipulador de solicitação em produção: cache → rota → rastreamento → execução → cache.

A aprovação humana é tratada pelo framework: porque `issue_refund` está com `approval_mode="always_require"`, a execução pausa e exibe um pedido de aprovação que resolvemos explicitamente.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Portão de Avaliação

Este é o portão de liberação da lição: um conjunto de testes offline avalia o agente, e o deployment só prossegue se a taxa de aprovação ultrapassar um limite. O avaliador aqui é uma simples verificação de sobreposição de palavras-chave para manter o notebook autocontido; em produção você usaria um LLM-como-juiz ou um avaliador de framework (veja a Lição 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Juntando Tudo: Um Lançamento Simulado

A célula abaixo mostra todo o loop descrito na lição: executar o portão de avaliação e somente "implantá-lo" se passar. Este é o padrão que você executaria em CI antes de promover uma versão do agente para o Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Resumo

Você montou um agente de suporte ao cliente pronto para produção com todas as preocupações operacionais integradas:

- **Ferramentas, RAG e memória** dão capacidade e contexto ao agente.
- **Roteamento e cache de modelo** mantêm a latência e o custo sob controle.
- **Aprovação humana** protege ações de alto risco, como grandes reembolsos.
- **O portão de avaliação** bloqueia lançamentos ruins antes que sejam enviados.
- **Rastreamento** torna cada solicitação observável.

### Desafio

Expanda este agente para:

1. **Suportar múltiplos modelos** — adicione uma terceira camada de "raciocínio" e encaminhe escalonamentos/reclamações para ela.
2. **Adicionar portões de avaliação** — expanda `TEST_CASES` para incluir cenários de aprovação de reembolso e confirme que o portão captura regressões.
3. **Adicionar roteamento consciente de custo** — acompanhe o custo estimado por solicitação (pequena vs grande vs cache) e imprima um relatório de custos após um lote de consultas mistas.

Na próxima lição você fará a jornada oposta e executará um agente inteiramente em sua própria máquina com Microsoft Foundry Local e Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
